## Arkansas Update SeaWulf Data with EI Analysis

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import dill

In [2]:
demographics = ['white', 'black', 'latino', 'other']
candidates = ['harris', 'trump']

### Import Models

In [3]:
path = '../output/Arkansas/models/ei_models.pkl'

In [4]:
with open(path, 'rb') as f:
    models = dill.load(f)

c:\Users\leeh1\Rockies-Gerrymandering\repo\rockies\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
c:\Users\leeh1\Rockies-Gerrymandering\repo\rockies\Lib\site-packages\numba\core\decorators.py:248: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [5]:
type(models)

dict

#### Get Sampled Shares Per Model 

In [5]:
def get_sample_shares(models):
    df = pd.DataFrame()

    for candidate in candidates:
        for demographic in demographics:
            ei = models[candidate][demographic]

            samples_matrix = ei.sim_trace["posterior"]["b_1"].stack(all_draws=["chain", "draw"]).values
            
            precinct_means = samples_matrix.mean(axis=1) 
            
            df[f'{demographic}_{candidate}_pct'] = precinct_means
    
    return df

In [6]:
sampled_shares = get_sample_shares(models)
sampled_shares

,white_harris_pct,black_harris_pct,latino_harris_pct,other_harris_pct,white_trump_pct,black_trump_pct,latino_trump_pct,other_trump_pct
0,0.089769,0.621649,0.615536,0.360707,0.897528,0.341397,0.313602,0.500113
1,0.249294,0.814320,0.614282,0.361601,0.708973,0.154308,0.314333,0.496840
2,0.063241,0.522248,0.611636,0.359300,0.924254,0.425539,0.314099,0.500015
3,0.386553,0.858957,0.612258,0.362059,0.571852,0.114763,0.314158,0.499677
4,0.041969,0.619011,0.609688,0.358003,0.952854,0.333626,0.319174,0.501522
...,...,...,...,...,...,...,...,...
2676,0.178865,0.811769,0.622826,0.363323,0.789686,0.157507,0.305731,0.492109
2677,0.095662,0.802553,0.597283,0.352292,0.888175,0.164551,0.327024,0.509665
2678,0.078158,0.798878,0.607704,0.334132,0.910896,0.167225,0.316878,0.530698
2679,0.148626,0.803538,0.613553,0.356545,0.828438,0.165049,0.316286,0.505002


### Update Seawulf Data

In [25]:
import geopandas
gdf = geopandas.read_file("../output/Arkansas/ar_seawulf.gpkg", driver="GPKG")
gdf.to_csv("../output/Arkansas/ar_seawulf.csv", index=False)
gdf

c:\Users\leeh1\Rockies-Gerrymandering\repo\rockies\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
c:\Users\leeh1\Rockies-Gerrymandering\repo\rockies\Lib\site-packages\pyogrio\geopandas.py:275: UserWarning: More than one layer found in 'ar_seawulf.gpkg': 'seawulf' (default), 'ar_seawulf'. Specify layer parameter to avoid this warning.
  result = read_func(


,Unique_ID,COUNTYFP,Kamala D. Harris,Donald J. Trump,Other_candidates,Total_votes,White_population,Black_population,Latino_population,Other_population,Total_population,geometry
0,05001-81 - Stuttgart 2,001,293,639,19,951,486.169395,356.604974,32.728840,0.738939,876.242147,"MULTIPOLYGON (((-91.52313 34.47881, -91.5234 3..."
1,05001-36 - Gillett Ward 3,001,25,39,2,66,63.524419,10.565301,0.982819,2.213607,77.286145,"MULTIPOLYGON (((-91.38939 34.1162, -91.38913 3..."
2,05001-53 - Dewitt 2,001,49,188,5,242,230.348940,133.055168,0.119868,4.781286,368.305262,"MULTIPOLYGON (((-91.32906 34.28234, -91.3327 3..."
3,05001-51 - DeWitt 1,001,101,61,5,167,332.713262,201.772191,0.000000,7.098115,541.583568,"MULTIPOLYGON (((-91.32906 34.28234, -91.32906 ..."
4,05001-55 - Dewitt 3,001,42,272,3,317,36.167855,9.010094,0.713321,0.281977,46.173247,"MULTIPOLYGON (((-91.34062 34.30203, -91.34062 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
2676,05149-10 - Dardanelle Ward 3,149,59,93,5,157,242.266509,33.762641,53.156670,23.501428,352.687248,"MULTIPOLYGON (((-93.15995 35.2133, -93.15995 3..."
2677,05149-18 - Magazine 1,149,79,354,7,440,763.771656,0.000000,69.264875,45.229594,878.266125,"MULTIPOLYGON (((-93.27993 35.21256, -93.27951 ..."
2678,05149-29 - Ions Creek,149,0,19,0,19,141.619378,2.686534,2.302779,13.816510,160.425201,"MULTIPOLYGON (((-93.70875 34.83304, -93.70873 ..."
2679,05149-24 - Waveland,149,18,69,1,88,253.901601,7.571889,4.456923,9.800575,275.730988,"MULTIPOLYGON (((-93.59987 35.14713, -93.59987 ..."


In [26]:
gdf = pd.concat([gdf, sampled_shares], axis=1)
gdf.columns = gdf.columns.str.capitalize()
gdf

,Unique_id,Countyfp,Kamala d. harris,Donald j. trump,Other_candidates,Total_votes,White_population,Black_population,Latino_population,Other_population,Total_population,Geometry,White_harris_pct,Black_harris_pct,Latino_harris_pct,Other_harris_pct,White_trump_pct,Black_trump_pct,Latino_trump_pct,Other_trump_pct
0,05001-81 - Stuttgart 2,001,293,639,19,951,486.169395,356.604974,32.728840,0.738939,876.242147,"MULTIPOLYGON (((-91.52313 34.47881, -91.5234 3...",0.089769,0.621649,0.615536,0.360707,0.897528,0.341397,0.313602,0.500113
1,05001-36 - Gillett Ward 3,001,25,39,2,66,63.524419,10.565301,0.982819,2.213607,77.286145,"MULTIPOLYGON (((-91.38939 34.1162, -91.38913 3...",0.249294,0.814320,0.614282,0.361601,0.708973,0.154308,0.314333,0.496840
2,05001-53 - Dewitt 2,001,49,188,5,242,230.348940,133.055168,0.119868,4.781286,368.305262,"MULTIPOLYGON (((-91.32906 34.28234, -91.3327 3...",0.063241,0.522248,0.611636,0.359300,0.924254,0.425539,0.314099,0.500015
3,05001-51 - DeWitt 1,001,101,61,5,167,332.713262,201.772191,0.000000,7.098115,541.583568,"MULTIPOLYGON (((-91.32906 34.28234, -91.32906 ...",0.386553,0.858957,0.612258,0.362059,0.571852,0.114763,0.314158,0.499677
4,05001-55 - Dewitt 3,001,42,272,3,317,36.167855,9.010094,0.713321,0.281977,46.173247,"MULTIPOLYGON (((-91.34062 34.30203, -91.34062 ...",0.041969,0.619011,0.609688,0.358003,0.952854,0.333626,0.319174,0.501522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2676,05149-10 - Dardanelle Ward 3,149,59,93,5,157,242.266509,33.762641,53.156670,23.501428,352.687248,"MULTIPOLYGON (((-93.15995 35.2133, -93.15995 3...",0.178865,0.811769,0.622826,0.363323,0.789686,0.157507,0.305731,0.492109
2677,05149-18 - Magazine 1,149,79,354,7,440,763.771656,0.000000,69.264875,45.229594,878.266125,"MULTIPOLYGON (((-93.27993 35.21256, -93.27951 ...",0.095662,0.802553,0.597283,0.352292,0.888175,0.164551,0.327024,0.509665
2678,05149-29 - Ions Creek,149,0,19,0,19,141.619378,2.686534,2.302779,13.816510,160.425201,"MULTIPOLYGON (((-93.70875 34.83304, -93.70873 ...",0.078158,0.798878,0.607704,0.334132,0.910896,0.167225,0.316878,0.530698
2679,05149-24 - Waveland,149,18,69,1,88,253.901601,7.571889,4.456923,9.800575,275.730988,"MULTIPOLYGON (((-93.59987 35.14713, -93.59987 ...",0.148626,0.803538,0.613553,0.356545,0.828438,0.165049,0.316286,0.505002


In [28]:
gdf.to_file("../output/Arkansas/ar_seawulf.gpkg", driver="GPKG")

In [29]:
gdf = geopandas.read_file("../output/Arkansas/ar_seawulf.gpkg", driver="GPKG")
gdf.to_csv("../output/Arkansas/ar_seawulf.csv", index=False)
gdf

c:\Users\leeh1\Rockies-Gerrymandering\repo\rockies\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


,Unique_id,Countyfp,Kamala d. harris,Donald j. trump,Other_candidates,Total_votes,White_population,Black_population,Latino_population,Other_population,Total_population,White_harris_pct,Black_harris_pct,Latino_harris_pct,Other_harris_pct,White_trump_pct,Black_trump_pct,Latino_trump_pct,Other_trump_pct,geometry
0,05001-81 - Stuttgart 2,001,293,639,19,951,486.169395,356.604974,32.728840,0.738939,876.242147,0.089769,0.621649,0.615536,0.360707,0.897528,0.341397,0.313602,0.500113,"MULTIPOLYGON (((-91.52313 34.47881, -91.5234 3..."
1,05001-36 - Gillett Ward 3,001,25,39,2,66,63.524419,10.565301,0.982819,2.213607,77.286145,0.249294,0.814320,0.614282,0.361601,0.708973,0.154308,0.314333,0.496840,"MULTIPOLYGON (((-91.38939 34.1162, -91.38913 3..."
2,05001-53 - Dewitt 2,001,49,188,5,242,230.348940,133.055168,0.119868,4.781286,368.305262,0.063241,0.522248,0.611636,0.359300,0.924254,0.425539,0.314099,0.500015,"MULTIPOLYGON (((-91.32906 34.28234, -91.3327 3..."
3,05001-51 - DeWitt 1,001,101,61,5,167,332.713262,201.772191,0.000000,7.098115,541.583568,0.386553,0.858957,0.612258,0.362059,0.571852,0.114763,0.314158,0.499677,"MULTIPOLYGON (((-91.32906 34.28234, -91.32906 ..."
4,05001-55 - Dewitt 3,001,42,272,3,317,36.167855,9.010094,0.713321,0.281977,46.173247,0.041969,0.619011,0.609688,0.358003,0.952854,0.333626,0.319174,0.501522,"MULTIPOLYGON (((-91.34062 34.30203, -91.34062 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2676,05149-10 - Dardanelle Ward 3,149,59,93,5,157,242.266509,33.762641,53.156670,23.501428,352.687248,0.178865,0.811769,0.622826,0.363323,0.789686,0.157507,0.305731,0.492109,"MULTIPOLYGON (((-93.15995 35.2133, -93.15995 3..."
2677,05149-18 - Magazine 1,149,79,354,7,440,763.771656,0.000000,69.264875,45.229594,878.266125,0.095662,0.802553,0.597283,0.352292,0.888175,0.164551,0.327024,0.509665,"MULTIPOLYGON (((-93.27993 35.21256, -93.27951 ..."
2678,05149-29 - Ions Creek,149,0,19,0,19,141.619378,2.686534,2.302779,13.816510,160.425201,0.078158,0.798878,0.607704,0.334132,0.910896,0.167225,0.316878,0.530698,"MULTIPOLYGON (((-93.70875 34.83304, -93.70873 ..."
2679,05149-24 - Waveland,149,18,69,1,88,253.901601,7.571889,4.456923,9.800575,275.730988,0.148626,0.803538,0.613553,0.356545,0.828438,0.165049,0.316286,0.505002,"MULTIPOLYGON (((-93.59987 35.14713, -93.59987 ..."


In [30]:
gdf.columns

Index(['Unique_id', 'Countyfp', 'Kamala d. harris', 'Donald j. trump',
       'Other_candidates', 'Total_votes', 'White_population',
       'Black_population', 'Latino_population', 'Other_population',
       'Total_population', 'White_harris_pct', 'Black_harris_pct',
       'Latino_harris_pct', 'Other_harris_pct', 'White_trump_pct',
       'Black_trump_pct', 'Latino_trump_pct', 'Other_trump_pct', 'geometry'],
      dtype='object')